# Phân tích A/B Testing (A/B Test Analysis) - Tuần 4
Mục tiêu: Đánh giá hiệu quả của chiến dịch Voucher (Treatment) lên nhóm khách hàng mục tiêu `Suburban Commuters` (Persona 0).

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

# Cài đặt giao diện
plt.style.use('ggplot')
sns.set_palette("Set2")

## 1. Tải và Lọc dữ liệu (Data Loading & Filtering)
Lọc tệp khách hàng mục tiêu: `Suburban Commuters` (Persona 0).

In [ ]:
# Tải dữ liệu
df = pd.read_csv('../../data/processed/segmented_simulation_data.csv')

# Khắc phục lỗi nếu Persona chưa được update hoàn chỉnh (như phát hiện lúc nãy)
if 'Persona A' in df['persona'].values:
    persona_names = {0: 'Suburban Commuters', 1: 'Urban Commuters', 2: 'Urban Leisure', 3: 'Suburban Occasionals'}
    df['persona'] = df['cluster_id'].map(persona_names)
    df.to_csv('../../data/processed/segmented_simulation_data.csv', index=False)

# Lọc nhóm mục tiêu
target_persona = 'Suburban Commuters'
df_target = df[df['persona'] == target_persona].copy()

print(f"Tổng số khách hàng trong nhóm {target_persona}: {len(df_target)}")
print(f"Nhóm Control: {len(df_target[df_target['treatment_rand'] == 0])} người")
print(f"Nhóm Treatment: {len(df_target[df_target['treatment_rand'] == 1])} người")

## 2. Sanity Check: Covariate Balance (Kiểm tra tính cân bằng)
Trước khi tin tưởng vào kết quả A/B Test, ta phải đảm bảo thuật toán chia ngẫu nhiên (Randomization) hoạt động đúng.
Nhóm Treatment và Control phải có đặc điểm (Covariates) tương đồng nhau trước khi thí nghiệm bắt đầu.

In [ ]:
def check_balance(df, feature):
    control = df[df['treatment_rand'] == 0][feature]
    treatment = df[df['treatment_rand'] == 1][feature]
    
    t_stat, p_val = stats.ttest_ind(control, treatment, equal_var=False)
    
    print(f"--- Kiểm tra cân bằng biến: {feature} ---")
    print(f"Control Mean: {control.mean():.2f}")
    print(f"Treatment Mean: {treatment.mean():.2f}")
    print(f"P-value: {p_val:.4f} -> {'CÂN BẰNG' if p_val > 0.05 else 'MẤT CÂN BẰNG (LỖI SRM)'}\n")

check_balance(df_target, 'age')
check_balance(df_target, 'monthly_rides_history')
check_balance(df_target, 'recency_days')

## 3. Phân tích OEC (Primary Metric): Incremental Rides
Đo lường tác động của Voucher lên số chuyến đi (`y_rand`).

In [ ]:
control_rides = df_target[df_target['treatment_rand'] == 0]['y_rand']
treatment_rides = df_target[df_target['treatment_rand'] == 1]['y_rand']

# Trực quan hóa
plt.figure(figsize=(10, 5))
sns.histplot(control_rides, color='blue', alpha=0.5, label='Control (Không Voucher)', kde=True, stat="density")
sns.histplot(treatment_rides, color='red', alpha=0.5, label='Treatment (Có Voucher)', kde=True, stat="density")
plt.title('Phân phối số chuyến xe giữa nhóm Treatment và Control')
plt.xlabel('Số chuyến (y_rand)')
plt.ylabel('Mật độ')
plt.legend()
plt.show()

# Tính ATE và T-Test
mean_control = control_rides.mean()
mean_treatment = treatment_rides.mean()
ate = mean_treatment - mean_control
relative_uplift = (ate / mean_control) * 100

t_stat, p_val = stats.ttest_ind(treatment_rides, control_rides, equal_var=False)

# Tính Confidence Interval 95%
se_control = control_rides.std() / np.sqrt(len(control_rides))
se_treatment = treatment_rides.std() / np.sqrt(len(treatment_rides))
se_diff = np.sqrt(se_control**2 + se_treatment**2)

margin_of_error = 1.96 * se_diff # Z-score 1.96 for 95% CI
ci_lower = ate - margin_of_error
ci_upper = ate + margin_of_error

print("=== KẾT QUẢ A/B TEST (OEC) ===")
print(f"Trung bình nhóm Control: {mean_control:.2f} chuyến/user")
print(f"Trung bình nhóm Treatment: {mean_treatment:.2f} chuyến/user")
print(f"Absolute Uplift (ATE): +{ate:.2f} chuyến/user")
print(f"Relative Uplift: +{relative_uplift:.2f}%")
print(f"95% Confidence Interval: [{ci_lower:.2f}, {ci_upper:.2f}]")
print(f"P-value: {p_val:.6f}")

if p_val < 0.05:
    print("\n🚀 KẾT LUẬN: Tác động mang ý nghĩa thống kê (Statistically Significant)!")
else:
    print("\n⚠️ KẾT LUẬN: Tác động KHÔNG có ý nghĩa thống kê (Not Statistically Significant).")

## 4. Phân tích Guardrail: Economics (ROI)
Tăng số chuyến là tốt, nhưng công ty có Lãi hay Lỗ?
Giả sử mỗi voucher tiêu tốn trung bình 5 USD (Discount Cost). Ta cần so sánh với doanh thu tăng thêm.

In [ ]:
control_fare = df_target[df_target['treatment_rand'] == 0]['fare_rand']
treatment_fare = df_target[df_target['treatment_rand'] == 1]['fare_rand']

mean_fare_control = control_fare.mean()
mean_fare_treatment = treatment_fare.mean()

incremental_gross_revenue = mean_fare_treatment - mean_fare_control
voucher_cost = 5.0 # Chi phí Voucher giả định trên 1 user

incremental_net_revenue = incremental_gross_revenue - voucher_cost
roi = incremental_net_revenue / voucher_cost

print("=== KẾT QUẢ KINH TẾ HỌC (ECONOMICS GUARDRAIL) ===")
print(f"Doanh thu trung bình nhóm Control: ${mean_fare_control:.2f}/user")
print(f"Doanh thu trung bình nhóm Treatment: ${mean_fare_treatment:.2f}/user")
print(f"Doanh thu gộp tăng thêm: +${incremental_gross_revenue:.2f}/user")
print(f"Chi phí phát hành Voucher: ${voucher_cost:.2f}/user")
print("-" * 40)
print(f"Doanh thu thuần tăng thêm (Net Revenue): ${incremental_net_revenue:.2f}/user")
print(f"ROI (Return on Investment): {roi*100:.1f}%")

if roi > 0:
    print("\n✅ ĐẠT YÊU CẦU: Chiến dịch có LÃI. Nên tiến hành Roll-out toàn hệ thống!")
else:
    print("\n❌ CẢNH BÁO: Chiến dịch bị LỖ. Cần giảm chi phí Voucher hoặc tối ưu lại Targeting.")